<a href="https://colab.research.google.com/github/vinods88/EVChargingOptimizerAgent/blob/main/ChargingOptimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Install and Setup
!pip install openai -q

import os
from openai import OpenAI
import json

# Paste your key here, or use Colab Secrets (recommended)
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("myopenaIKey")

# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"
client = OpenAI()
print("Client ready.")

Client ready.


In [12]:
# Define the Tools
import random
import requests
import json
from datetime import datetime

def get_vehicle_state():
    """Returns current battery state of the EV."""
    return {"soc_pct": 40, "plugged_in": True,
            "battery_kwh": 75, "target_soc_pct": 80}
def get_electricity_prices(args: dict) -> dict:
    """
    Fetch electricity prices from Elpriset API
    args should contain:
    - date: "YYYY-MM-DD" (optional, defaults to today)
    - region: "SE1", "SE2", "SE3", or "SE4" (optional, defaults to "SE3" for Stockholm)
    """
    date = args.get("date", datetime.now().strftime("%Y-%m-%d"))
    region = args.get("region", "SE3")  # SE3 = Stockholm (your location)

    # Parse the date
    try:
        dt = datetime.strptime(date, "%Y-%m-%d")
        year = dt.strftime("%Y")
        month = dt.strftime("%m")
        day = dt.strftime("%d")
    except ValueError:
        return {"error": f"Invalid date format. Use YYYY-MM-DD"}

    # Build the API URL
    url = f"https://www.elprisetjustnu.se/api/v1/prices/{year}/{month}-{day}_{region}.json"

    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        prices = response.json()

        # Format the response with hourly prices
        hourly_prices = []
        for price_obj in prices:
            # Extract hour from timestamp (format: "2026-05-14T12:00:00+02:00")
            try:
                hour_str = price_obj["time_start"].split('T')[1].split(':')[0]
                hour = int(hour_str)
                hourly_prices.append({
                    "hour": hour,
                    "price_sek_per_kwh": price_obj["SEK_per_kWh"]
                })
            except (KeyError, IndexError, ValueError):
                # Fallback if format is different
                hourly_prices.append({
                    "hour": len(hourly_prices),
                    "price_sek_per_kwh": price_obj.get("SEK_per_kWh", 0)
                })

        avg_price = sum(p["price_sek_per_kwh"] for p in hourly_prices) / len(hourly_prices) if hourly_prices else 0

        # Format the response nicely
        return {
            "date": date,
            "region": region,
            "hourly_prices": hourly_prices,
            "average_sek_per_kwh": round(avg_price, 4),
            "min_price_sek_per_kwh": round(min(p["price_sek_per_kwh"] for p in hourly_prices), 4),
            "max_price_sek_per_kwh": round(max(p["price_sek_per_kwh"] for p in hourly_prices), 4)
        }
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to fetch prices: {str(e)}"}

#The below function is a mock function created initially to store price per hour. Ignore this
#def get_grid_tariff(hours: int):
#    """Returns simulated electricity prices for next N hours."""
#    base = [2.1, 1.8, 1.2, 0.8, 0.7, 0.9,
#            1.4, 2.5, 3.2, 3.5, 3.1, 2.8,
#            2.4, 2.2, 2.0, 2.1, 2.9, 3.4,
#            3.6, 3.8, 3.2, 2.6, 2.3, 2.0]
#    return [{"hour": i, "price_sek_per_kwh": base[i % 24]} for i in range(hours)]

# def get_user_schedule():
#     """Returns when the user needs the car next."""
#     return {"next_departure": "07:00",
#             "hours_until_departure": 7,
#             "flexible": False}


# def set_charging_schedule(start_hour: int, end_hour: int, rate_kw: float):
#    """Sends charging plan to EVSE (simulated)."""
#    energy = rate_kw * (end_hour - start_hour)
#    cost = energy * 0.85
#    return {"confirmed": True, "start_hour": start_hour,
#            "end_hour": end_hour, "rate_kw": rate_kw,
#            "estimated_cost_sek": round(cost, 2)}





In [13]:
#Tool definitions (tells the LLM what tools exist)
tools = [
  {"type": "function", "function": {
    "name": "get_vehicle_state",
    "description": "Get current EV battery state: SoC, plug status, capacity.",
    "parameters": {"type": "object", "properties": {}, "required": []}}},


 { "type": "function",
        "function": {
            "name": "get_electricity_prices",
            "description": "Fetch Swedish electricity prices (spotpris) from Elpriset API. Returns hourly prices for a specific date and region in SEK per kWh.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string",
                        "description": "Date in YYYY-MM-DD format (e.g., 2026-05-11). Defaults to today."
                    },
                    "region": {
                        "type": "string",
                        "enum": ["SE1", "SE2", "SE3", "SE4"],
                        "description": "Swedish price region: SE1=Luleå, SE2=Sundsvall, SE3=Stockholm, SE4=Malmö. Defaults to SE3."
                    }
                },
                "required": []
            }
        }
 },


  # {"type": "function", "function": {
  #   "name": "get_user_schedule",
  #   "description": "Get user's next departure time and schedule flexibility.",
  #   "parameters": {"type": "object", "properties": {}, "required": []}}},

  # {"type": "function", "function": {
  #   "name": "set_charging_schedule",
  #   "description": "Send the final charging plan to the EVSE.",
  #   "parameters": {"type": "object",
  #     "properties": {
  #       "start_hour": {"type": "integer", "description": "Hour to start (0-23)"},
  #       "end_hour":   {"type": "integer", "description": "Hour to stop (0-23)"},
  #       "rate_kw":    {"type": "number",  "description": "Charge rate in kW (max 11)"}},
  #     "required": ["start_hour", "end_hour", "rate_kw"]}}}
]

In [14]:
# The Agent Loop:
TOOL_MAP = {
    "get_vehicle_state":    lambda _: get_vehicle_state(),
    "get_electricity_prices" : get_electricity_prices,
    # "get_user_schedule":    lambda _: get_user_schedule(),
    # "set_charging_schedule":lambda a: set_charging_schedule(**a),
}

SYSTEM_PROMPT = """You are an EV charging optimizer agent.
Your job: decide the optimal charging schedule given vehicle state,
electricity prices, and the user's departure time.
Rules: never let SoC drop below 20%. Max charger rate is 11 kW.
Calculate how much it would cost based the energy used and the spot price. Let the user know about the cost
Explain your decision in plain language at the end."""

def run_agent(user_request: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_request}
    ]

    print(f"\n--- Agent starting ---")
    for step in range(10):
        response = client.chat.completions.create(
            model="gpt-5.5",
            messages=messages,
            tools=tools
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  Tool call: {tc.function.name}({args})")
                result = TOOL_MAP[tc.function.name](args)
                print(f"  Result:    {result}")
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result)
                })
        else:
            print(f"\n--- Agent decision ---\n{msg.content}")
            return msg.content

    return "Agent did not finish within 10 steps."

The agent loop in Cell 4 is the heart of everything — it keeps calling the LLM until there are no more t**ool_calls** in the response. That's the ReAct pattern: reason, act, observe result, repeat.
The TOOL_MAP dictionary is how tool names (strings from the LLM) get dispatched to real Python functions. This is the bridge between the LLM world and your code.
The three commented-out scenarios in Cell 5 are your first tests — try all three and verify the agent behaves sensibly for each. The "15% battery, leaving in 1 hour" case in particular should trigger immediate charging regardless of tariff prices.

In [15]:
# Try different scenarios:
#run_agent("My EV is plugged in. Optimize my charging for tonight.")

run_agent("I need to leave by 6am tomorrow from Skåne. Charge as cheaply as possible.")
# run_agent("Battery is at 15%. I need to leave in 1 hour — charge now.")


--- Agent starting ---
  Tool call: get_vehicle_state({})
  Result:    {'soc_pct': 40, 'plugged_in': True, 'battery_kwh': 75, 'target_soc_pct': 80}
  Tool call: get_electricity_prices({'date': '2026-05-14', 'region': 'SE4'})
  Result:    {'date': '2026-05-14', 'region': 'SE4', 'hourly_prices': [{'hour': 0, 'price_sek_per_kwh': 1.54325}, {'hour': 0, 'price_sek_per_kwh': 1.45644}, {'hour': 0, 'price_sek_per_kwh': 1.37989}, {'hour': 0, 'price_sek_per_kwh': 1.32366}, {'hour': 1, 'price_sek_per_kwh': 1.38764}, {'hour': 1, 'price_sek_per_kwh': 1.33436}, {'hour': 1, 'price_sek_per_kwh': 1.28184}, {'hour': 1, 'price_sek_per_kwh': 1.24908}, {'hour': 2, 'price_sek_per_kwh': 1.30935}, {'hour': 2, 'price_sek_per_kwh': 1.27648}, {'hour': 2, 'price_sek_per_kwh': 1.25945}, {'hour': 2, 'price_sek_per_kwh': 1.23226}, {'hour': 3, 'price_sek_per_kwh': 1.25224}, {'hour': 3, 'price_sek_per_kwh': 1.23969}, {'hour': 3, 'price_sek_per_kwh': 1.21719}, {'hour': 3, 'price_sek_per_kwh': 1.21555}, {'hour': 4, 'pr

'Your EV is plugged in in Skåne / SE4.\n\n- Current SoC: 40%\n- Target SoC: 80%\n- Battery size: 75 kWh\n- Energy needed: 30 kWh\n- Max charge rate: 11 kW\n- Charging time needed: about 2 h 44 min\n- Departure: 06:00 tomorrow\n\n### Cheapest charging plan\n\nCharge today during the very cheap afternoon price window:\n\n| Time | Action |\n|---|---|\n| 12:46–15:30 | Charge at up to 11 kW |\n| 15:30–06:00 | Stop charging / stay plugged in |\n\nThis gets the car to about 80% before your 06:00 departure.\n\n### Estimated cost\n\nUsing the cheapest SE4 spot-price periods available before departure:\n\n- Energy added: 30 kWh\n- Estimated spot-price cost: ≈ 2.33 SEK\n\nThis estimate is based on spot price only, excluding grid fees, taxes, VAT, and charging losses.\n\n### Plain-language explanation\n\nThe car needs 30 kWh to go from 40% to 80%. At 11 kW, that takes just under 3 hours. The cheapest electricity before your departure is not overnight — it is this afternoon in Skåne — so the cheape